In [11]:
from dolfinx import *
from mpi4py import MPI
import numpy as np
import matplotlib.pyplot as plt
import ufl
import scipy.io
from petsc4py import PETSc
import basix

mesh = mesh.create_unit_cube(MPI.COMM_WORLD, nx=1, ny=1, nz=1)
n = ufl.FacetNormal(mesh)

# On the main branch of DOLFINx, ufl.VectorElement has been replaced with basix.ufl.element(family, cell, degree, shape=(N,))
# to create a N-dimensional vector space for a given Finite element family.

P = basix.ufl.element("Lagrange", mesh.basix_cell(), 1, shape=(mesh.geometry.dim,)) # Vector?
R = basix.ufl.element("Lagrange", mesh.basix_cell(), 1, discontinuous=True) # Tensor?
P_wish = ufl.VectorElement("CG", mesh.ufl_cell(), 1)
R_wish = ufl.TensorElement("DG", mesh.ufl_cell(), 1)
mel = ufl.MixedElement([P, R, P_wish, R_wish])
print(mel)
U = fem.FunctionSpace(mesh, mel)

u = fem.Function(U)
(p, F, v, P) = ufl.split(u)
(dv, dP, dp, dF) = ufl.TestFunctions(U)

rho_0=1
mu=1
lamb=1

Psi_fun=lambda F: mu/2*(np.trace(F.T@F)-3)-mu*np.log(np.sqrt(np.linalg.det(F.T@F)))+lamb/2*np.log(np.sqrt(np.linalg.det(F.T@F)))**2
P_fun=lambda F: ufl.dot(F,mu*(ufl.classes.Identity(F.ufl_shape[0])-ufl.inv(ufl.dot(F.T,F)))+lamb*ufl.ln(ufl.sqrt(ufl.classes.Determinant(ufl.dot(F.T,F))))*ufl.inv(ufl.dot(F.T,F)))

u_n = fem.Function(U) # previous time step
(p_n, F_n, v_n, P_n) = ufl.split(u_n)

# Initial conditions:

def initial_cond(x,y,z):
    return np.eye(F.ufl_shape[0])
F.interpolate(np.eye(F.ufl_shape[0]))
F_n.interpolate(np.eye(F.ufl_shape[0]))

AttributeError: module 'ufl' has no attribute 'VectorElement'